# الفصل 7: بناء تطبيقات الدردشة
## بدء سريع مع واجهة برمجة التطبيقات لنماذج Microsoft Foundry

تم تكييف هذا الدفتر من [مستودع عينات Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) الذي يتضمن دفاتر تصل إلى خدمات [Azure OpenAI](notebook-azure-openai.ipynb).

> **ملاحظة:** تنهي نماذج GitHub خدماتها في نهاية يوليو 2026. الآن يستهدف هذا الدفتر [نماذج Microsoft Foundry](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst)، التي تقدم نفس كتالوج النماذج المجاني للتجربة وتجربة SDK استدلال Azure AI.


# نظرة عامة  
"نماذج اللغة الكبيرة هي دوال تقوم بتحويل النص إلى نص. عند إعطاء نص إدخال، يحاول نموذج اللغة الكبير التنبؤ بالنص الذي سيأتي بعده"(1). سيقدم هذا الدفتر "البداية السريعة" للمستخدمين مفاهيم عالية المستوى لنماذج اللغة الكبيرة، ومتطلبات الحزمة الأساسية للبدء مع AML، ومقدمة بسيطة لتصميم المطالبات، والعديد من الأمثلة القصيرة لاستخدامات مختلفة. 


## جدول المحتويات  

[نظرة عامة](#overview)  
[كيفية استخدام خدمة OpenAI](#how-to-use-openai-service)  
[1. إنشاء خدمتك في OpenAI](#1.-creating-your-openai-service)  
[2. التثبيت](#2.-installation)    
[3. بيانات الاعتماد](#3.-credentials)  

[حالات الاستخدام](#use-cases)    
[1. تلخيص النص](#1.-summarize-text)  
[2. تصنيف النص](#2.-classify-text)  
[3. توليد أسماء منتجات جديدة](#3.-generate-new-product-names)  
[4. تعديل دقيق لمصنف](#4.fine-tune-a-classifier)  

[المراجع](#references)


### أنشئ طلبك الأول  
ستوفر هذه التمرين القصير مقدمة أساسية لإرسال الطلبات إلى نموذج في Microsoft Foundry Models لمهمة بسيطة "التلخيص".


**الخطوات**:  
1. قم بتثبيت مكتبة `azure-ai-inference` في بيئة بايثون الخاصة بك، إذا لم تكن قد فعلت ذلك بعد.  
2. قم بتحميل مكتبات المساعدة القياسية وضبط بيانات اعتماد Microsoft Foundry Models الخاصة بك.  
3. اختر نموذجًا لمهمتك  
4. أنشئ طلبًا بسيطًا للنموذج  
5. قدم طلبك إلى واجهة برمجة تطبيقات النموذج!


### 1. تثبيت `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. استيراد مكتبات المساعدة وإنشاء بيانات الاعتماد


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### ٣. إيجاد النموذج المناسب  
يمكن لنماذج مثل GPT-4o و GPT-4o mini فهم وتوليد اللغة الطبيعية، وهي متاحة في كتالوج نماذج Microsoft Foundry بجانب نماذج من Meta و Mistral و Cohere و Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. تصميم الموجه  

"السحر في النماذج اللغوية الكبيرة هو أنه من خلال التدريب على تقليل خطأ التنبؤ هذا عبر كميات هائلة من النصوص، تنتهي النماذج بتعلم مفاهيم مفيدة لهذه التنبؤات. على سبيل المثال، يتعلمون مفاهيم مثل"(1):

* كيفية التهجئة
* كيفية عمل القواعد النحوية
* كيفية إعادة الصياغة
* كيفية الإجابة على الأسئلة
* كيفية إجراء محادثة
* كيفية الكتابة بالعديد من اللغات
* كيفية البرمجة
* إلخ.

#### كيفية التحكم في نموذج لغوي كبير  
"من بين جميع المدخلات إلى نموذج لغوي كبير، الموجه النصي هو الأكثر تأثيرًا بفارق كبير(1).

يمكن تحفيز النماذج اللغوية الكبيرة لإنتاج مخرجات بعدة طرق:

التعليمات: أخبر النموذج بما تريد
الإكمال: حث النموذج على إكمال بداية ما تريد
العرض: أرِ النموذج ما تريد، إما بـ:
بعض الأمثلة في الموجه
مئات أو آلاف الأمثلة في مجموعة بيانات تدريبية لضبط دقيق"



#### هناك ثلاث إرشادات أساسية لإنشاء الموجهات:

**العرض والإخبار**. اجعل ما تريد واضحًا إما من خلال التعليمات، أو الأمثلة، أو مزيج من الاثنين. إذا كنت تريد من النموذج ترتيب قائمة من العناصر بترتيب أبجدي أو تصنيف فقرة حسب المشاعر، أرِه هذا هو ما تريده.

**توفير بيانات ذات جودة**. إذا كنت تحاول بناء مصنف أو ترغب في أن يتبع النموذج نمطًا معينًا، تأكد من وجود عدد كافٍ من الأمثلة. تأكد من مراجعة أمثلتك — النموذج عادة ذكي بما يكفي لرؤية الأخطاء الإملائية الأساسية وتقديم رد، لكنه قد يفترض أيضًا أن هذا مقصود مما قد يؤثر على الرد.

**تحقق من إعداداتك.** تتحكم إعدادات درجة الحرارة و top_p في مدى حتمية النموذج في توليد الرد. إذا كنت تطلب منه ردًا حيث يوجد جواب واحد صحيح فقط، فسترغب في ضبط هذه القيم منخفضة. إذا كنت تبحث عن ردود أكثر تنوعًا، فربما تود رفعها. الخطأ الأول الذي يقع فيه الناس مع هذه الإعدادات هو افتراض أنها تحكم "الذكاء" أو "الإبداع".


المصدر: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. قدم!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### كرر نفس المكالمة، كيف تقارن النتائج؟


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## تلخيص النص  
#### التحدي  
قم بتلخيص النص عن طريق إضافة 'tl;dr:' إلى نهاية المقطع النصي. لاحظ كيف يفهم النموذج كيفية أداء عدد من المهام دون تعليمات إضافية. يمكنك تجربة مطالبات أكثر وصفية من tl;dr لتعديل سلوك النموذج وتخصيص التلخيص الذي تتلقاه(3).  

أظهرت الأعمال الحديثة تحقيق مكاسب كبيرة في العديد من مهام ومعايير معالجة اللغة الطبيعية من خلال التدريب المسبق على مجموعة كبيرة من النصوص يليه ضبط دقيق لمهمة محددة. بينما يكون الهيكل عادةً غير مخصص لمهمة معينة، إلا أن هذه الطريقة لا تزال تتطلب مجموعات بيانات ضبط دقيق خاصة بالمهمة تحتوي على آلاف أو عشرات الآلاف من الأمثلة. بالمقابل، يمكن للبشر عادةً أداء مهمة لغوية جديدة بناءً على عدد قليل من الأمثلة أو تعليمات بسيطة - وهو أمر لا تزال أنظمة معالجة اللغة الطبيعية الحالية تكافح إلى حد كبير للقيام به. هنا نُظهر أن توسيع نماذج اللغة يحسن بشكل كبير الأداء غير الخاص بالمهمة وعدد اللقطات القليل، وأحيانًا يصل إلى التنافسية مع طرائق الضبط الدقيق المتقدمة السابقة. 



ملخص  


# تمارين لعدة استخدامات  
1. تلخيص النص  
2. تصنيف النص  
3. توليد أسماء منتجات جديدة


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## تصنيف النص  
#### التحدي  
تصنيف العناصر إلى فئات يتم توفيرها في وقت الاستدلال. في المثال التالي، نوفر كل من الفئات والنص المراد تصنيفه في الموجه(*playground_reference). 

استعلام العميل: مرحبًا، تعطلت مؤخرًا إحدى مفاتيح لوحة مفاتيح اللابتوب الخاص بي وسأحتاج إلى بديل:

الفئة المصنفة:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## إنشاء أسماء منتجات جديدة
#### التحدي
إنشاء أسماء منتجات من كلمات الأمثلة. هنا نقوم بإدراج معلومات في النص حول المنتج الذي سنقوم بإنشاء أسماء له. كما نقدم مثالًا مشابهًا لعرض النمط الذي نرغب في استلامه. لقد قمنا أيضًا بتعيين قيمة درجة الحرارة عالية لزيادة العشوائية والحصول على ردود مبتكرة أكثر.

وصف المنتج: صانع مخفوق الحليب للمنزل
كلمات البداية: سريع، صحي، مضغوط.
أسماء المنتج: HomeShaker, Fit Shaker, QuickShake, Shake Maker

وصف المنتج: زوج من الأحذية يمكنه أن يناسب أي مقاس قدم.
كلمات البداية: قابل للتكيف، مناسب، يناسب الجميع.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# المراجع  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [بوابة Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [أفضل الممارسات لضبط نماذج GPT لتصنيف النصوص](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# للمزيد من المساعدة  
[فريق التسويق في OpenAI](AzureOpenAITeam@microsoft.com) 


# المساهمون
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
